# 05 — Functional Features, Lambdas, Walrus Operator, and Custom Sorting

## Anonymous Functions: Lambda Expressions

In computer science, **lambda expressions** trace their origins to Alonzo Church's $\lambda$-calculus. In Python, a lambda expression creates an anonymous function object:

$$\text{lambda } x_1, x_2, \dots, x_n : \text{expression}$$

### Syntactic and Semantic Constraints
1. **Single Expression Only:** A lambda can evaluate only a single expression; it cannot contain statements (`return`, `raise`, `while`, assignments).
2. **Implicit Return:** The evaluated result of the expression is returned automatically.
3. **Primary Use Cases:** Ephemeral callback functions passed to higher-order functions (`sorted`, `map`, `filter`).


In [ ]:
# 1. Lambda Expressions and Closures
# Passing an ephemeral lambda to a higher-order transformation function
def transform_value(x: float, operation) -> float:
    """Applies a callable operation to a scalar value."""
    return operation(x)


result = transform_value(8.0, lambda x: x**2)
print(f"Transformed value via lambda: {result}")
assert result == 64.0


# Factory function returning a closure with lambda
def make_polynomial(a: float, b: float, c: float):
    """Returns quadratic polynomial function f(x) = a*x^2 + b*x + c."""
    return lambda x: a * (x**2) + b * x + c


f_poly = make_polynomial(2.0, -3.0, 1.0)
assert f_poly(2.0) == 2.0 * (4.0) - 6.0 + 1.0  # 8 - 6 + 1 = 3.0
print(f"Evaluated polynomial closure at x=2.0: {f_poly(2.0)}")

## Custom Sorting with Key Callables and Timsort

Python's built-in `sorted()` and `list.sort()` employ the **Timsort** algorithm (a hybrid of merge sort and insertion sort). Timsort is **stable**: it preserves the relative order of elements that compare equal.

The `key` parameter accepts a callable that extracts a comparison key from each element before sorting.


In [ ]:
# 2. Multi-Criteria Sorting with Lambda Keys
# Dataset of student model evaluations: (model_name, validation_f1, latency_ms)
evaluations = [
    ("Model-A", 0.89, 45),
    ("Model-B", 0.94, 120),
    ("Model-C", 0.94, 85),
    ("Model-D", 0.82, 20),
]

# Sort primarily by validation_f1 DESCENDING, tie-breaker by latency_ms ASCENDING
ranked_models = sorted(evaluations, key=lambda record: (-record[1], record[2]))

print("Ranked Models (Highest F1, then lowest latency):")
for rank, (name, f1, lat) in enumerate(ranked_models, start=1):
    print(f"  {rank}. {name}: F1={f1:.2f}, Latency={lat}ms")

assert ranked_models[0] == ("Model-C", 0.94, 85)
assert ranked_models[1] == ("Model-B", 0.94, 120)

## Sorting Dictionaries by Keys and Values

Because dictionaries are composed of key-value pairs, sorting can be applied to keys, values, or items.


In [ ]:
# 3. Sorting Dictionaries
epoch_losses = {
    "epoch_1": 0.65,
    "epoch_2": 0.42,
    "epoch_3": 0.28,
    "epoch_4": 0.31,
    "epoch_5": 0.19,
}

# Sorting items by value (loss ascending)
sorted_by_loss = sorted(epoch_losses.items(), key=lambda kv: kv[1])
best_epoch, lowest_loss = sorted_by_loss[0]

print(f"Best performance: {best_epoch} with loss={lowest_loss}")
assert best_epoch == "epoch_5" and lowest_loss == 0.19

## The Walrus Operator (`:=` — Assignment Expressions)

Introduced in PEP 572 (Python 3.8), the **walrus operator** (`:=`) allows assigning values to variables inside expressions.

### Critical Advantages
- **Eliminating redundant computations:** Avoids computing expensive expressions twice (once in a condition, once in the body).
- **Concise stream processing:** Simplifies loop boundaries.


In [ ]:
# 4. The Walrus Operator in Comprehensions and Filters
tokens = ["convolution", "ai", "transformer", "nlp", "recurrent", "cv"]

# Filtering and transforming simultaneously in a comprehension
# Compute uppercase only once, and filter where length exceeds 3
uppercase_long_tokens = [upper_tok for tok in tokens if len(upper_tok := tok.upper()) > 3]

print(f"Filtered uppercase tokens: {uppercase_long_tokens}")
assert uppercase_long_tokens == ["CONVOLUTION", "TRANSFORMER", "RECURRENT"]

# Simulating stream reading with walrus
data_stream = ["header: init", "metric: 0.95", "metric: 0.97", "DONE"]
stream_iter = iter(data_stream)
collected_metrics = []

while (line := next(stream_iter, None)) and line != "DONE":
    if line.startswith("metric:"):
        collected_metrics.append(float(line.split(":")[1].strip()))

print(f"Collected metrics via walrus loop: {collected_metrics}")
assert collected_metrics == [0.95, 0.97]

## Type Annotations for Compound Structures (PEP 585 & PEP 604)

Modern Python incorporates type hints directly into built-in collection types without requiring imports from `typing` (e.g., `list[tuple[str, float]]` instead of `typing.List[typing.Tuple[str, float]]`).


In [ ]:
# 5. Type Hints with Built-In Generics
from collections.abc import Callable


def filter_and_rank(
    records: list[tuple[str, float]],
    min_score: float,
    transform_fn: Callable[[float], float] | None = None,
) -> list[tuple[str, float]]:
    """Filters records by min_score, applies optional transform, and ranks descending."""
    results: list[tuple[str, float]] = []
    for label, score in records:
        if score >= min_score:
            final_score = transform_fn(score) if transform_fn else score
            results.append((label, final_score))

    return sorted(results, key=lambda item: item[1], reverse=True)


sample_data = [("cat", 0.95), ("dog", 0.40), ("bird", 0.82)]
ranked = filter_and_rank(sample_data, min_score=0.5, transform_fn=lambda s: round(s * 100, 1))

print(f"Ranked and scaled: {ranked}")
assert ranked == [("cat", 95.0), ("bird", 82.0)]